# 6 — Station high-SWE vs. low-SWE backscatter comparison (manuscript Fig. A1)

Recreates the manuscript's Fig. A1 within the `compare_to_all_public_snow_pillows` framework: for a single station, a **high-SWE water year** and a **low-SWE water year** side by side —

- **Top:** daily snow pillow SWE with the 95%-of-maximum threshold and the snow pillow runoff onset (from the framework's `max_snow_pillow_swe_timing.zarr`)
- **Middle:** Sentinel-1 VV backscatter per relative orbit, the MODIS snow appearance / disappearance dates, the phenology-constrained backscatter-minimum search window (midpoint of the snow-covered period → disappearance + 16 d), per-orbit minima, and their median (the SAR runoff onset)
- **Bottom:** the signed timing offset between the two estimates

Successor of `compare_to_snotel/supplemental_figure_methodology.ipynb` (which produced the manuscript's Fig. A1, `figures/v9/high_swe_low_swe_backscatter_comparison_846_CA_SNTL_2017_2021.png`, from the SNOTEL-only comparison dataset). Inputs here are this framework's zarrs (`data/snow_pillows/`, `data/comparison_datasets/<version>/`) plus a fresh Sentinel-1 RTC fetch at the station; the figure is written to `figures/<version>/`.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.dates as mdates
from matplotlib.legend_handler import HandlerBase
import odc.geo.xr  # registers the .odc accessor

from global_snowmelt_runoff_onset.config import Config
import global_snowmelt_runoff_onset.processing as processing

warnings.filterwarnings('ignore')

In [ ]:
from pathlib import Path

config = Config('config/global_config_v10.txt')

# Figures and the input comparison datasets are scoped by dataset version so a
# rerun against another version cannot overwrite these outputs. Switching
# versions = editing the config path above.
VERSION = config.version
DATA_DIR = Path('data')
FIGURE_DIR = Path('figures') / VERSION
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

snow_pillows_ds = xr.open_zarr(DATA_DIR / 'snow_pillows' / 'snow_pillows.zarr')
swe_timing_ds = xr.open_zarr(DATA_DIR / 'comparison_datasets' / VERSION / 'max_snow_pillow_swe_timing.zarr')
chips_ds = xr.open_zarr(DATA_DIR / 'comparison_datasets' / VERSION / 'runoff_onset_snow_pillow_station_chips.zarr')

## Candidate scan (optional)

Rank stations that have both a clean high-SWE year and a clean low-SWE year, using only the framework's precomputed SWE timing dataset. Purely to suggest `station_id` / water-year choices for the settings cell below — skip if you already know what to plot.

In [ ]:
max_swe_m = (swe_timing_ds['max_SWE_value'] / 1000).compute()        # mm -> m
sp_timing = swe_timing_ds['95pct_of_max_SWE_timing'].compute()        # DOWY

HIGH_SWE_MIN = 0.60   # m
LOW_SWE_MAX  = 0.30   # m
LOW_SWE_MIN  = 0.15   # m — below ~0.15 m the pillow onset itself gets ambiguous

records = []
for sid in max_swe_m.station_id.values:
    swe_s = max_swe_m.sel(station_id=sid)
    tim_s = sp_timing.sel(station_id=sid)
    valid = np.isfinite(swe_s.values) & np.isfinite(tim_s.values)
    if valid.sum() < 3:
        continue
    wys = swe_s.water_year.values[valid]
    swe_v = swe_s.values[valid]
    hi_mask = swe_v >= HIGH_SWE_MIN
    lo_mask = (swe_v >= LOW_SWE_MIN) & (swe_v <= LOW_SWE_MAX)
    if not (hi_mask.any() and lo_mask.any()):
        continue
    hi_i = np.argmax(swe_v)                       # highest-SWE year
    lo_i = np.where(lo_mask)[0][np.argmin(swe_v[lo_mask])]  # lowest qualifying year
    records.append(dict(station_id=str(sid), high_swe_wy=int(wys[hi_i]),
                        high_swe_m=round(float(swe_v[hi_i]), 2),
                        low_swe_wy=int(wys[lo_i]),
                        low_swe_m=round(float(swe_v[lo_i]), 2),
                        swe_contrast=round(float(swe_v[hi_i] - swe_v[lo_i]), 2)))

candidates = pd.DataFrame(records).set_index('station_id').sort_values('swe_contrast', ascending=False)
print(f'{len(candidates)} candidate stations')
candidates.head(15)

In [ ]:
# ── USER SETTINGS ─────────────────────────────────────────────────────────────
station_id    = '846_CA_SNTL'   # Virginia Lakes Ridge — the manuscript Fig. A1 station
high_swe_wy   = 2017
low_swe_wy    = 2021
buffer_radius = 1000            # m — matches the manuscript's evaluation radius
# ──────────────────────────────────────────────────────────────────────────────

station_timing = swe_timing_ds.sel(station_id=station_id).compute()
station_name = str(station_timing.station_name.values)
station_elev = float(station_timing.elevation.values)
print(f'{station_name} ({station_id}), {station_elev:.0f} m — '
      f'WY{high_swe_wy} max SWE {float(station_timing.max_SWE_value.sel(water_year=high_swe_wy))/1000:.2f} m, '
      f'WY{low_swe_wy} max SWE {float(station_timing.max_SWE_value.sel(water_year=low_swe_wy))/1000:.2f} m')

## Station SWE, geometry, Sentinel-1 backscatter, and snow phenology

In [ ]:
# Daily SWE series (mm -> m) from the framework's snow pillow archive
swe_m = (snow_pillows_ds['swe'].sel(station_id=station_id) / 1000).compute()

def wy_slice(wy):
    """Time-slice strings for a (northern hemisphere) water year."""
    return f'{wy-1}-10-01', f'{wy}-09-30'

def dowy_to_date(dowy, wy):
    """Convert a day-of-water-year value to a calendar date."""
    return pd.Timestamp(year=wy - 1, month=10, day=1) + pd.Timedelta(days=float(dowy) - 1)

In [ ]:
# Station point -> local UTM -> buffered square bbox, like the framework's chips
station_gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy([float(station_timing.longitude)],
                                [float(station_timing.latitude)]),
    crs='EPSG:4326')
station_gdf = station_gdf.to_crs(station_gdf.estimate_utm_crs())
station_gdf['geometry'] = station_gdf.geometry.buffer(buffer_radius)
station_gdf

In [ ]:
# Sentinel-1 VV backscatter for the buffered station box: clip the runoff onset
# store to get the exact 80 m grid, then fetch the full S1 RTC record on that
# geobox and reduce to one series per acquisition (median over the box, in dB)
runoff_onset_global_ds = config.open_runoff_onset_dataset(chunks=None)
runoff_onset_station_ds = runoff_onset_global_ds.rio.clip_box(
    *station_gdf.total_bounds, crs=station_gdf.crs
).rio.reproject(station_gdf.crs)

s1_rtc_ds = processing.get_sentinel1_rtc(runoff_onset_station_ds.odc.geobox,
                                         start_date='2014-10-01')
backscatter_da = s1_rtc_ds.median(dim=['x', 'y']).compute()
vv_dB = 10 * np.log10(backscatter_da)['vv']
print(f'{vv_dB.sizes["time"]} S1 acquisitions loaded.')

In [ ]:
# MODIS snow phenology at the station: median SAD/SDD DOWY over the buffered box
# (configs >= v10 use an icechunk Zarr v3 phenology store, <= v9 consolidated Zarr v2)
if config.snow_phenology_store_is_icechunk:
    snow_phenology_ds = xr.open_zarr(config.snow_phenology_store, zarr_format=3,
                                     consolidated=False, decode_coords='all')
else:
    snow_phenology_ds = xr.open_zarr(config.snow_phenology_store,
                                     consolidated=True, decode_coords='all')

snow_phenology_clip_ds = snow_phenology_ds.rio.clip_box(
    *station_gdf.total_bounds, crs=station_gdf.crs
).compute()

SAD_DOWY = snow_phenology_clip_ds['SAD_DOWY'].median(dim=['x', 'y'])
SDD_DOWY = snow_phenology_clip_ds['SDD_DOWY'].median(dim=['x', 'y'])

SAD_dates = {int(wy): dowy_to_date(float(SAD_DOWY.sel(water_year=wy)), int(wy))
             for wy in SAD_DOWY.water_year.values
             if np.isfinite(float(SAD_DOWY.sel(water_year=wy)))}
SDD_dates = {int(wy): dowy_to_date(float(SDD_DOWY.sel(water_year=wy)), int(wy))
             for wy in SDD_DOWY.water_year.values
             if np.isfinite(float(SDD_DOWY.sel(water_year=wy)))}
print({wy: (SAD_dates[wy].strftime('%b %d'), SDD_dates[wy].strftime('%b %d'))
       for wy in (high_swe_wy, low_swe_wy)})

## Onset computation (mirrors the dataset methodology)

Snow pillow onset comes straight from the framework's `95pct_of_max_SWE_timing` (the day SWE drops below 95% of the water-year maximum, as computed by `1_create_snow_pillow_comparison_dataset.ipynb`). The SAR onset is recomputed here from the raw backscatter: per relative orbit, the minimum VV backscatter within the search window (midpoint of the snow-covered period → snow disappearance + 16 d), then the median date across orbits — the same logic the production pipeline applies per pixel.

In [ ]:
SDD_OFFSET = pd.Timedelta(days=16)
ORBIT_COLORS = ["#0400ff", "#0046c7", "#0084ff", '#fdae6b', '#9ecae1']

def get_sar_for_wy(vv_dB, wy):
    start, end = (pd.Timestamp(t) for t in wy_slice(wy))
    times = pd.to_datetime(vv_dB.time.values)
    return vv_dB.isel(time=(times >= start) & (times <= end))

def compute_sar_onset(vv_wy_da, sad_date, sdd_date):
    """Per-orbit backscatter minima within the phenology-constrained search
    window, and their median date (the SAR runoff onset estimate)."""
    if sad_date is None or sdd_date is None:
        return None, np.nan, []

    sdd_adjusted  = sdd_date + SDD_OFFSET
    snow_midpoint = sad_date + (sdd_date - sad_date) / 2   # midpoint of the snow-covered period
    times         = pd.to_datetime(vv_wy_da.time.values)
    vals          = vv_wy_da.values
    orbit_results = []

    for i, rel_orbit in enumerate(np.unique(vv_wy_da['sat:relative_orbit'].values)):
        mask_orbit  = (vv_wy_da['sat:relative_orbit'] == rel_orbit).values
        sub_times   = times[mask_orbit]
        sub_vals    = vals[mask_orbit]
        window_mask = (sub_times >= snow_midpoint) & (sub_times <= sdd_adjusted)
        if not np.any(window_mask) or np.all(np.isnan(sub_vals[window_mask])):
            continue
        w_times = sub_times[window_mask]
        w_vals  = sub_vals[window_mask]
        min_idx = np.nanargmin(w_vals)
        orbit_results.append({'orbit': rel_orbit, 'date': w_times[min_idx],
                              'dB': w_vals[min_idx],
                              'color': ORBIT_COLORS[i % len(ORBIT_COLORS)]})

    if not orbit_results:
        return None, np.nan, []

    ordinals    = [d['date'].toordinal() for d in orbit_results]
    median_ord  = int(np.median(ordinals))
    median_date = pd.Timestamp.fromordinal(median_ord)
    closest     = min(orbit_results, key=lambda d: abs(d['date'].toordinal() - median_ord))
    return median_date, closest['dB'], orbit_results

In [ ]:
def build_year(wy):
    r = {'wy': wy}
    r['swe'] = swe_m.sel(time=slice(*wy_slice(wy)))
    r['vv']  = get_sar_for_wy(vv_dB, wy)
    r['sad'] = SAD_dates.get(wy)
    r['sdd'] = SDD_dates.get(wy)

    r['max_swe']   = float(station_timing['max_SWE_value'].sel(water_year=wy)) / 1000
    r['threshold'] = 0.95 * r['max_swe']
    sp_dowy = float(station_timing['95pct_of_max_SWE_timing'].sel(water_year=wy))
    r['sp_onset'] = dowy_to_date(sp_dowy, wy) if np.isfinite(sp_dowy) else None

    r['sar_onset'], r['min_dB'], r['orbit_results'] = compute_sar_onset(
        r['vv'], r['sad'], r['sdd'])
    r['diff_days'] = ((r['sar_onset'] - r['sp_onset']).days
                      if r['sp_onset'] is not None and r['sar_onset'] is not None
                      else None)
    return r

r_high = build_year(high_swe_wy)
r_low  = build_year(low_swe_wy)

# sanity check against the published product: median runoff onset over the chip
for r in (r_high, r_low):
    chip_onset_dowy = float(chips_ds['runoff_onset']
                            .sel(station_id=station_id, water_year=r['wy'])
                            .compute().median())
    chip_onset = (dowy_to_date(chip_onset_dowy, r['wy']).strftime('%b %d')
                  if np.isfinite(chip_onset_dowy) else 'n/a')
    print(f"WY {r['wy']}: pillow onset {r['sp_onset']:%b %d} | "
          f"SAR onset (this notebook) {r['sar_onset']:%b %d} | "
          f"dataset chip median onset {chip_onset} | "
          f"offset (SAR - pillow) {r['diff_days']:+d} d")

## Build the figure

Layout: **2 columns × 3 rows** — column 1 = high-SWE year, column 2 = low-SWE year; row 1 = snow pillow SWE (95% threshold, onset marked), row 2 = per-orbit VV backscatter (phenology dates, search window, per-orbit minima, median), row 3 = timing comparison.

In [ ]:
# ── Colours ───────────────────────────────────────────────────────────────────
COL_FILTERED = "#fc8a5e"
COL_THRESH   = "#55514a"
COL_SP       = '#d73027'
COL_SAR      = '#111111'
COL_SAD      = "#389912"
COL_SDD      = "#C01A9C"

# ── Custom legend handler: vertical line + optional triangle ──────────────────
class HandlerVLine(HandlerBase):
    """Renders a vertical line (with optional triangle marker) in the legend key."""
    def __init__(self, color, lw=0.5, ls='-', alpha=1.0,
                 marker=None, ms=5, marker_alpha=1.0, marker_pos=0.35):
        self._color, self._lw, self._ls, self._alpha = color, lw, ls, alpha
        self._marker, self._ms = marker, ms
        self._marker_alpha, self._marker_pos = marker_alpha, marker_pos
        super().__init__()

    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height, fontsize, trans):
        cx = xdescent + width / 2.0
        artists = []
        vline_bg = plt.Line2D([cx, cx], [ydescent - 2, ydescent + height + 2],
                              color='white', linewidth=self._lw + 1.0,
                              linestyle=self._ls, solid_capstyle='butt',
                              transform=trans)
        artists.append(vline_bg)
        vline = plt.Line2D([cx, cx], [ydescent - 2, ydescent + height + 2],
                           color=self._color, linewidth=self._lw,
                           linestyle=self._ls, alpha=self._alpha,
                           solid_capstyle='butt', transform=trans)
        artists.append(vline)
        if self._marker:
            my = ydescent + height * self._marker_pos
            artists.append(plt.Line2D([cx], [my], marker=self._marker,
                                      color=self._color, markersize=7,
                                      alpha=self._marker_alpha, linestyle='None',
                                      markeredgecolor='white', markeredgewidth=0.6,
                                      transform=trans))
        return artists


def vline_entry(label, color, lw=1.5, ls='-', alpha=1.0,
                marker=None, ms=5, marker_alpha=1.0, marker_pos=0.35):
    proxy = mlines.Line2D([], [], color=color, lw=lw, linestyle=ls,
                          alpha=alpha, label=label)
    handler = HandlerVLine(color=color, lw=lw, ls=ls, alpha=alpha,
                           marker=marker, ms=ms,
                           marker_alpha=marker_alpha, marker_pos=marker_pos)
    return proxy, handler


def apply_xaxis_fmt(ax, show_labels=True):
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    if show_labels:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)
    else:
        ax.xaxis.set_major_formatter(mdates.DateFormatter(''))

In [ ]:
# ── Panel functions ───────────────────────────────────────────────────────────
def plot_swe_panel(ax, r, right=False):
    wy      = r['wy']
    xlim    = (pd.Timestamp(f'{wy-1}-10-01'), pd.Timestamp(f'{wy}-09-30'))
    times_f = pd.to_datetime(r['swe'].time.values)

    ax.plot(times_f, r['swe'].values, color=COL_FILTERED, lw=1.8, zorder=2)
    ax.axhline(r['threshold'], color=COL_THRESH, lw=1.3, ls='--', zorder=3)

    if r['sp_onset'] is not None:
        ax.axvline(r['sp_onset'], color=COL_SP, lw=2.0, zorder=4)
        ax.plot(r['sp_onset'], r['max_swe'] + 0.05, 'v', color=COL_SP, ms=9, zorder=5)

    sp_proxy, sp_handler = vline_entry(
        label=f"Snow pillow onset "
              f"({r['sp_onset'].strftime('%b %d') if r['sp_onset'] else 'N/A'})",
        color=COL_SP, lw=2.0, marker='v', ms=4, marker_pos=0.50)
    handles = [
        mlines.Line2D([], [], color=COL_FILTERED, lw=1.8, label='SWE'),
        mlines.Line2D([], [], color=COL_THRESH, lw=1.3, ls='--',
                      label=f"95% of max SWE ({r['threshold']:.2f} m)"),
        sp_proxy,
    ]
    ax.legend(handles=handles, handler_map={sp_proxy: sp_handler},
              fontsize=8, loc='upper left', framealpha=0.85)
    ax.set_xlim(xlim)
    ax.set_ylim(bottom=0)
    ax.set_ylabel('SWE (m)', fontsize=10)
    ax.grid(True, alpha=0.3)
    apply_xaxis_fmt(ax, show_labels=False)
    if right:
        ax.yaxis.set_label_position('right')
        ax.yaxis.tick_right()


def plot_sar_panel(ax, r, right=False):
    wy           = r['wy']
    xlim         = (pd.Timestamp(f'{wy-1}-10-01'), pd.Timestamp(f'{wy}-09-30'))
    vv           = r['vv']
    sad          = r['sad']
    sdd          = r['sdd']
    sdd_adjusted = r['sdd'] + SDD_OFFSET if r['sdd'] is not None else None

    # per-orbit time series -> right legend
    orbit_handles = []
    for i, rel_orbit in enumerate(np.unique(vv['sat:relative_orbit'].values)):
        sub         = vv.isel(time=(vv['sat:relative_orbit'] == rel_orbit))
        orbit_state = sub['sat:orbit_state'].values[0]
        tod         = 'ascending, afternoon' if orbit_state == 'ascending' \
                      else 'descending, morning'
        color       = ORBIT_COLORS[i % len(ORBIT_COLORS)]
        ax.plot(pd.to_datetime(sub.time.values), sub.values,
                color=color, lw=1.2, alpha=0.85, zorder=2)
        orbit_handles.append(mlines.Line2D([], [], color=color, lw=1.2,
                                           label=f'Orbit {rel_orbit} ({tod})'))
    leg_right = ax.legend(handles=orbit_handles, fontsize=8,
                          loc='lower right', framealpha=0.85)
    ax.add_artist(leg_right)

    # per-orbit minima: dotted vlines + triangles
    for orb in r['orbit_results']:
        ax.axvline(orb['date'], color=orb['color'], lw=1.4, ls=':',
                   alpha=0.65, zorder=3)
        ax.plot(orb['date'], orb['dB'], '^', color=orb['color'], ms=9,
                alpha=0.70, zorder=4, markeredgewidth=0.6, markeredgecolor='white')

    # median minimum: bold black vline + larger triangle
    if r['sar_onset'] is not None:
        ax.axvline(r['sar_onset'], color=COL_SAR, lw=2.4, zorder=5)
        ax.plot(r['sar_onset'], r['min_dB'] - 1.0, '^', color=COL_SAR, ms=11,
                zorder=6, markeredgewidth=0.5, markeredgecolor='white')

    # left legend: MODIS dates + search window + per-orbit + median
    left_handles, left_handlers = [], {}
    if sad is not None:
        ax.axvline(sad, color=COL_SAD, lw=2.2, ls='--', zorder=3)
        p, h = vline_entry(f'Snow appearance ({sad.strftime("%b %d")})',
                           COL_SAD, lw=2.5, ls='--')
        left_handles.append(p); left_handlers[p] = h
    if sdd is not None:
        ax.axvline(sdd, color=COL_SDD, lw=2.2, ls='--', zorder=3)
        p, h = vline_entry(
            f'Snow disappearance ({sdd.strftime("%b %d")})',
            COL_SDD, lw=2.5, ls='--')
        left_handles.append(p); left_handlers[p] = h
    if sad is not None and sdd is not None:
        snow_midpoint = sad + (sdd - sad) / 2
        ax.axvspan(snow_midpoint, sdd_adjusted, color='steelblue', alpha=0.12, zorder=1)
        left_handles.append(mpatches.Patch(facecolor='steelblue', alpha=0.35,
                                           label=f'Temporal search window ({snow_midpoint.strftime("%b %d")}-{sdd_adjusted.strftime("%b %d")})'))

    p_orb, h_orb = vline_entry('Per-orbit backscatter min timing',
                               '#666666', lw=2.0, ls=':', alpha=0.75,
                               marker='^', ms=4, marker_alpha=0.8, marker_pos=0.50)
    left_handles.append(p_orb); left_handlers[p_orb] = h_orb

    if r['sar_onset'] is not None:
        p_med, h_med = vline_entry(
            f"Median backscatter min timing ({r['sar_onset'].strftime('%b %d')})",
            COL_SAR, lw=2.0, marker='^', ms=6, marker_pos=0.50)
        left_handles.append(p_med); left_handlers[p_med] = h_med

    ax.legend(handles=left_handles, handler_map=left_handlers,
              fontsize=7, loc='lower left', framealpha=0.85)
    ax.add_artist(leg_right)

    ax.set_xlim(xlim)
    ax.set_ylabel('VV backscatter (dB)', fontsize=10)
    ax.grid(True, alpha=0.3)
    apply_xaxis_fmt(ax, show_labels=False)
    if right:
        ax.yaxis.set_label_position('right')
        ax.yaxis.tick_right()


def plot_timeline_panel(ax, r, right=False):
    wy   = r['wy']
    xlim = (pd.Timestamp(f'{wy-1}-10-01'), pd.Timestamp(f'{wy}-09-30'))

    ax.set_xlim(xlim)
    ax.set_yticks([])
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='x')

    y_sp, y_sar = 0.65, 0.35
    LABEL_OFFSET = pd.Timedelta(days=10)

    if r['sp_onset'] is not None:
        ax.axvline(r['sp_onset'], color=COL_SP, lw=2.5, zorder=4)
        ax.plot(r['sp_onset'], y_sp, 'v', color=COL_SP, ms=9, zorder=5)
        label_x = r['sp_onset'] + LABEL_OFFSET
        ax.text(label_x, y_sp + 0.16, 'Snow pillow onset',
                ha='left', va='bottom', fontsize=9, color=COL_SP, fontweight='bold')
        ax.text(label_x, y_sp + 0.04, r['sp_onset'].strftime('%b %d'),
                ha='left', va='bottom', fontsize=9, color=COL_SP, fontweight='bold')

    if r['sar_onset'] is not None:
        ax.axvline(r['sar_onset'], color=COL_SAR, lw=2.5, zorder=5)
        ax.plot(r['sar_onset'], y_sar, '^', color=COL_SAR, ms=11, zorder=6,
                markeredgewidth=0.5, markeredgecolor='white')
        label_x = r['sar_onset'] - LABEL_OFFSET
        ax.text(label_x, y_sar - 0.03, 'Median backscatter min timing',
                ha='right', va='top', fontsize=9, color=COL_SAR, fontweight='bold')
        ax.text(label_x, y_sar - 0.15, r['sar_onset'].strftime('%b %d'),
                ha='right', va='top', fontsize=9, color=COL_SAR, fontweight='bold')

    if r['diff_days'] is not None:
        box_x = xlim[1] - pd.Timedelta(days=10)
        ax.text(box_x, 0.08, f"timing offset of {r['diff_days']} days",
                ha='right', va='bottom', fontsize=10, color='#333333',
                bbox=dict(boxstyle='round,pad=0.3', fc='white',
                          ec='#aaaaaa', alpha=0.9))

    apply_xaxis_fmt(ax, show_labels=True)
    ax.set_xlabel('Date', fontsize=10)
    if right:
        ax.yaxis.set_label_position('right')

In [ ]:
# ── Figure layout: 3 rows x 2 cols ────────────────────────────────────────────
fig = plt.figure(figsize=(13, 9))
gs = gridspec.GridSpec(nrows=3, ncols=2, figure=fig,
                       hspace=0.10, wspace=0.08,
                       height_ratios=[2.0, 2.0, 0.85])

ax_swe_h = fig.add_subplot(gs[0, 0])
ax_swe_l = fig.add_subplot(gs[0, 1])
ax_sar_h = fig.add_subplot(gs[1, 0])
ax_sar_l = fig.add_subplot(gs[1, 1])
ax_tl_h  = fig.add_subplot(gs[2, 0])
ax_tl_l  = fig.add_subplot(gs[2, 1])

plot_swe_panel(ax_swe_h, r_high, right=False)
plot_swe_panel(ax_swe_l, r_low,  right=True)
swe_ymax = max(ax_swe_h.get_ylim()[1], ax_swe_l.get_ylim()[1])
ax_swe_h.set_ylim(0, swe_ymax + 0.12)   # headroom for the onset triangle
ax_swe_l.set_ylim(0, swe_ymax + 0.12)

plot_sar_panel(ax_sar_h, r_high, right=False)
plot_sar_panel(ax_sar_l, r_low,  right=True)
sar_ymin = min(ax_sar_h.get_ylim()[0], ax_sar_l.get_ylim()[0]) - 2
sar_ymax = max(ax_sar_h.get_ylim()[1], ax_sar_l.get_ylim()[1])
ax_sar_h.set_ylim(sar_ymin, sar_ymax)
ax_sar_l.set_ylim(sar_ymin, sar_ymax)

plot_timeline_panel(ax_tl_h, r_high, right=False)
plot_timeline_panel(ax_tl_l, r_low,  right=True)

ax_swe_h.set_title(f'High-SWE year (WY {high_swe_wy})', fontsize=12, fontweight='bold')
ax_swe_l.set_title(f'Low-SWE year (WY {low_swe_wy})',   fontsize=12, fontweight='bold')

#fig.suptitle(f'{station_id} SWE vs. backscatter time series')

stem = f'high_swe_low_swe_backscatter_comparison_{station_id}_{high_swe_wy}_{low_swe_wy}'
plt.savefig(FIGURE_DIR / f'{stem}.png', dpi=350, bbox_inches='tight', facecolor='white')  # manuscript Fig. A1 styling
plt.show()
print(f'saved {FIGURE_DIR / (stem + ".png")}')